## nn.Linear layer version

## Try now with EEG simulations

In [1]:
from pydeconv.utils import *
from pydeconv.pydeconv_sims import *
import numpy as np
import mne

%matplotlib qt 
n_seconds = 10000        # Duration of the signal in seconds
sfreq = 500            # Sampling frequency in Hz
sig = EEGSimulator(n_seconds, sfreq)
# transition probabilities
W_matrix = [[0, 0.45, 0.45, 0.1],[0.9, 0, 0 , .1],[0.9, 0, 0,.1], [.33,.33,.33,0]]
kernels = {
            0: {'onsets': [0, 0.19, 0.25], 'amplitudes': [0.1, -0.05, 0.04], 'widths': [0.05, 0.05, 0.07], 'weight':0.3},
            1: {'onsets': [0, 0.19, 0.25], 'amplitudes': [0.1, -0.05, 0.04], 'widths': [0.05, 0.05, 0.07], 'weight':0.3},
            2: {'onsets': [0, 0.19, 0.25], 'amplitudes': [0.1, -0.07, 0.04], 'widths': [0.05, 0.05, 0.07], 'weight':0.3},
            3: {'onsets': [0, 0.19, 0.25], 'amplitudes': [0.1, -0.07, 0.04], 'widths': [0.05, 0.05, 0.07], 'weight':0.3},
            'modulation': {'ker_idx2mod': 1, 'mod': 'linear','dist': 'uniform', 'lims': [100, 600]}
        }

sig.create_isi_pdf(0, sample_size=100, lims=[.01, .15], dist_type='skewed', mode=.05, skew=0, scale=.01)
sig.create_isi_pdf(1, sample_size=100, lims=[.1, .6], dist_type='skewed', mode=.3, skew=2, scale=.05)
sig.create_isi_pdf(2, sample_size=100, lims=[.1, .6], dist_type='skewed', mode=.3, skew=2, scale=.05)
sig.create_isi_pdf(3, sample_size=100, lims=[.1, .6], dist_type='uniform')


# sig.combine_isi_pdf
# sig.plot_isi_pdf(0)
# sig.plot_isi_pdf(1)

################
sig.simulate(noise=None,erp_ker=kernels,w_matrix=W_matrix)

# create evts

# Copy and modify the event data
evts = sig.evts.copy()

# Set the event type to filter (event_id 1 for example)
event_id1 = 1
event_id2 = 2

# Filter events where `type == event_id`
filtered_evts = evts.loc[(evts['type'] == event_id1) | (evts['type'] == event_id2)]

# Get the number of filtered events
n_events = len(filtered_evts)

# Ensure that latencies are integer values
latencies = filtered_evts['latency'].values.astype(int)

# Create the events array for MNE
# Column 1: Latencies
# Column 2: Zeros (assuming no previous event values, hence zeros)
# Column 3: Event types (all set to 1 since filtered for `event_id`)
mne_events = np.column_stack((latencies,
                              np.zeros(n_events, dtype=int),
                              np.ones(n_events, dtype=int)))

# Print or use `mne_events` as needed
print(mne_events[:5])
#create RAW
# Creating simulated RAW

# Parameters

n_samples = n_seconds * sfreq  # Total number of samples
n_channels = 1         # Number of channels (virtual channel)

# Create random data for the virtual channel (shape: [n_samples])
data = sig.data

# Reshape the data to be 2D (n_channels, n_samples)
data = data.reshape((n_channels, n_samples))

# Create MNE info object
ch_names = ['VirtualEEG']  # Name of the virtual channel
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')

# Create the Raw object from the reshaped data array
raw = mne.io.RawArray(data, info)

evts['type'].value_counts()

# Update 'effect' column
evts['effect'] = evts['type'].apply(lambda x: True if x == 2 else False if x == 1 else np.nan)

# Replace all 2s with 1s in 'type' column
evts['type'] = evts['type'].replace(2, 1)
evts['type'].value_counts()
columns = {'latencies':latencies,	'type':'type','categorical':'categorical','continuous':'continuous'}
# evts.rename(columns=columns, inplace=True)



No noise added
[[ 10   0   1]
 [324   0   1]
 [384   0   1]
 [450   0   1]
 [655   0   1]]
Creating RawArray with float64 data, n_channels=1, n_times=5000000
    Range : 0 ... 4999999 =      0.000 ...  9999.998 secs
Ready.


In [5]:
# Load parameter, data and features
#==================================
from pydeconv import *
import config4test
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error
settings = analyze_data(config4test)
features = evts
# raw     = raw
# Initialize the model
#=====================
rERP_model = PyDeconv(settings = settings , features = features, eeg = raw)
X_design = rERP_model.create_matrix()
y_data   = rERP_model.get_nonzero_data()

# Model Selection 
#================
solver = rERP_model.estimator
num_folds = 5
alphas = np.linspace(5, 500, 20)
param_grid = {'alpha': alphas.tolist()}
# Create StratifiedKFold object
kf = KFold(n_splits=num_folds)
# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=solver, 
                           param_grid=param_grid, 
                           scoring='neg_mean_squared_error', 
                           cv=kf,verbose=5)
grid_search.fit(X_design, y_data)
# rERP_model.estimator.set_params(alpha = 24)
# rERP_model.fit(X_design, y_data)

# # Extract results
# #================
cv_results = grid_search.cv_results_
best_model = grid_search.best_estimator_
rERP_model.coef_ = best_model.coef_

# fig = rERP_model.plot_coefs()

# rERP_model.coef_.shape
rERP_model.plot_coefs(top_topos=False)
plt.show()


Analyzing data with model: testingSims
Time range: -0.2 to 0.6
Solver: ridge

Model Name: testingSims
First Intercept Event Type: 1
Second Intercept Event Type: 0
Sampling Frequency: 500.0
Time Window: -0.2 to 0.6
Channels to Analyze: 1

Model Description:
Intercept: True
Additive Features: ['effect']
Interactions: None


Original Design Matrix Shape:
X_design shape: (5000000, 1203)
y_data shape: (5000000, 1)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END ........................alpha=5.0;, score=-0.042 total time=   0.1s
[CV 2/5] END ........................alpha=5.0;, score=-0.046 total time=   0.1s
[CV 3/5] END ........................alpha=5.0;, score=-0.039 total time=   0.1s
[CV 4/5] END ........................alpha=5.0;, score=-0.040 total time=   0.1s
[CV 5/5] END ........................alpha=5.0;, score=-0.043 total time=   0.1s
[CV 1/5] END ..........alpha=31.05263157894737;, score=-0.042 total time=   0.0s
[CV 2/5] END ..........alpha=31.052631

In [2]:
# X_design.shape
data.shape

(1, 5000000)

In [3]:
# After GridSearchCV
# Debugging: Print the shape of X_design and expected feature count
print("Shape of X_design:", X_design.shape)
print("Expected features from feature_names:", (rERP_model.n_predictors) * rERP_model.n_delays)

# Fit the model if the shapes match
if X_design.shape[1] == (rERP_model.n_predictors * rERP_model.n_delays):
    rERP_model.fit(X_design, y_data)
else:
    print("Feature mismatch: check feature_names and design matrix setup.")

best_model = grid_search.best_estimator_
rERP_model.estimator = best_model

# Fit the model with the selected parameters
rERP_model.fit(X_design, y_data)  # Ensure estimator_ is properly initialized

print("Shape of X_design:", X_design.shape)

# Now you can use the score function
score = rERP_model.score(X_design, y_data)
print("Model score:", score)

NameError: name 'X_design' is not defined

In [67]:

# GPU FIT WITH DATALOADER
#========
# rERP_model = PyDeconv(settings = settings , features = features, eeg = raw)


# PyTorch Ridge Model
# torch_ridge = Ridge(alpha=1e-3, fit_intercept=True, batch_size=1000, device=device)
# torch_ridge.fit(X_design, y_data, epochs=30, lr=0.01)
torch_ridge = Ridge(input_dim=X_design.shape[1], output_dim= y_data.shape[1], alpha= 5 , fit_intercept=False, batch_size=100000, device=device)
torch_ridge.fit(X_design, y_data, epochs=50, lr=0.005, momentum=0,scheduler_gamma=.001)

# Plot learning curve
torch_ridge.plot_learning_curve()
coeffs =  torch_ridge.linear.weight.detach().cpu().numpy()
rERP_model.coef_ =  coeffs
# # Predictions should now have shape (13000, 3)
# torch_predictions = torch_ridge.predict(X_torch)
rERP_model.plot_coefs(top_topos=False)
plt.show()

TypeError: Ridge.fit() got an unexpected keyword argument 'momentum'

In [6]:
import pandas as pd
import mne
from pydeconv.utils import *
from pydeconv import *
from sklearn.model_selection import KFold, GridSearchCV
import torch
import torch.nn as nn 
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

# Check for available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Ridge:
    def __init__(self, input_dim, output_dim, alpha=1.0, fit_intercept=True, batch_size=32, device='cpu'):
        self.alpha = alpha
        self.fit_intercept = fit_intercept
        self.batch_size = batch_size
        self.device = device
        self.loss_history = []
        
        # Define the nn.Linear layer with the fit_intercept parameter controlling bias
        self.linear = nn.Linear(input_dim, output_dim, bias=fit_intercept).to(device)

    def fit(self, X, y, epochs=1000, lr=0.01, momentum=0, scheduler_step_size=10, scheduler_gamma=0.1):
        optimizer = optim.SGD(self.linear.parameters(), lr=lr, momentum=momentum)
        
        # Scheduler to reduce learning rate at regular intervals
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_step_size, gamma=scheduler_gamma)

        # Convert sparse matrix to dense format
        X_dense = X.todense() if hasattr(X, "todense") else X
        y_dense = y.toarray() if hasattr(y, "toarray") else y

        # Convert data to tensors
        X_tensor = torch.tensor(X_dense, dtype=torch.float32).to("cpu")
        y_tensor = torch.tensor(y_dense, dtype=torch.float32).to("cpu")
        dataset = TensorDataset(X_tensor, y_tensor)

        # Training loop
        for epoch in range(epochs):
            # Shuffle dataset by creating a new DataLoader every epoch
            data_loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

            epoch_loss = 0.0
            for X_batch, y_batch in data_loader:
                # Move mini-batch to GPU
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                
                # Forward pass
                predictions = self.model(X_batch)
                loss = self.loss(predictions, y_batch)
                epoch_loss += loss.item()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Step the scheduler after each epoch
            scheduler.step()

            avg_loss = epoch_loss / len(data_loader)
            self.loss_history.append(avg_loss)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4e}, LR: {scheduler.get_last_lr()[0]:.6f}")

    def model(self, x):
        return self.linear(x)

    def loss(self, predictions, y):
        mse_loss = torch.mean((predictions - y) ** 2)
        ridge_penalty = self.alpha * torch.sum(self.linear.weight ** 2)
        return mse_loss + ridge_penalty

    def predict(self, X):
        # Convert input to tensor and move to device
        X_dense = X.todense() if hasattr(X, "todense") else X
        X_tensor = torch.tensor(X_dense, dtype=torch.float32).to(self.device)
        return self.model(X_tensor).detach().cpu().numpy()

    def plot_learning_curve(self):
        plt.plot(self.loss_history, label="Training Loss")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.title("Learning Curve")
        plt.legend()
        plt.show()

# # Load parameters, data, and features here
# # data_path = "./example_data/"
# # settings = analyze_data()
# # features = pd.read_csv(data_path + "629959_full_metadata.csv") 
# # raw     = mne.io.read_raw_eeglab(data_path + "629959_analysis.set", preload=True)

# # Initialize and train the model
# # rERP_model = PyDeconv(settings=settings, features=features, eeg=raw)
# # X_design = rERP_model.create_matrix()
# # y_data = rERP_model.get_nonzero_data()
# torch_ridge = Ridge(input_dim=X_design.shape[1], output_dim=y_data.shape[1], alpha=5, fit_intercept=False, batch_size=200000, device=device)
# torch_ridge.fit(X_design, y_data, epochs=20, lr=0.05, momentum=0.3)

# # Plot learning curve
# torch_ridge.plot_learning_curve()


In [44]:
rERP_model

<PyDeconv | tmin, tmax : (-0.200, 0.600), estimator : <class 'sklearn.linear_model._ridge.Ridge'>, features : [intercept, ..., 5], fit: True>

In [21]:
X_design.shape

(57777, 1203)

In [14]:
features['effect'].unique()

array([nan, True, False], dtype=object)

In [19]:
coeffs.shape

(1, 1203)

In [74]:
grid_search.best_estimator_

Ridge(alpha=5.0)

In [7]:
# Check for available device
device = ('cuda' if torch.cuda.is_available() else 'cpu')

class Ridge:
    def __init__(self, input_dim, output_dim, alpha=1.0, fit_intercept=True, batch_size=32, device='cpu'):
        self.alpha = alpha
        self.fit_intercept = fit_intercept
        self.batch_size = batch_size
        self.device = device
        self.loss_history = []
        
        # Define the nn.Linear layer with the fit_intercept parameter controlling bias
        self.linear = nn.Linear(input_dim, output_dim, bias=fit_intercept).to(device)

    def fit(self, X, y, epochs=1000, lr=0.01, momentum=0):
        n_samples = X.shape[0]
        
        optimizer = optim.SGD(self.linear.parameters(), lr=lr, momentum=momentum )


        # Convert data to CPU tensors only (avoiding full GPU memory usage initially)
        X_tensor = torch.tensor(X.todense(), dtype=torch.float32)  # Keep on CPU
        y_tensor = torch.tensor(y, dtype=torch.float32)  # Keep on CPU
        dataset = TensorDataset(X_tensor, y_tensor)
        data_loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        # Training loop
        for epoch in range(epochs):
            epoch_loss = 0.0
            for X_batch, y_batch in data_loader:
                # Move mini-batch to GPU
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                
                # Forward pass
                predictions = self.model(X_batch)
                loss = self.loss(predictions, y_batch)
                epoch_loss += loss.item()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            avg_loss = epoch_loss / len(data_loader)
            self.loss_history.append(avg_loss)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


            

    def model(self, x):
        return self.linear(x)

    def loss(self, predictions, y):
        mse_loss = torch.mean((predictions - y) ** 2)
        ridge_penalty = self.alpha * torch.sum(self.linear.weight ** 2)
        return mse_loss + ridge_penalty

    def predict(self, X):
        return self.model(X).detach().cpu().numpy()

    def plot_learning_curve(self):
        plt.plot(self.loss_history, label="Training Loss")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.title("Learning Curve")
        plt.legend()
        plt.show()

In [64]:
evts

,latency,type,categorical,continuous,effect
0,0.000000,0,NaN,NaN,NaN
1,6.507745,1,NaN,NaN,False
2,55.250512,0,NaN,NaN,NaN
3,58.813516,1,NaN,NaN,True
4,131.136892,0,NaN,NaN,NaN
...,...,...,...,...,...
1195,59235.908146,1,NaN,NaN,True
1196,59301.063636,0,NaN,NaN,NaN
1197,59300.275653,1,NaN,NaN,True
1198,59361.545860,0,NaN,NaN,NaN


In [73]:
plt.plot(rERP_model.coef_.T)

In [60]:
plt.plot(coeffs.T)

## playground

In [8]:
import torch
import torch.optim as optim
import torch.nn as nn
import matplotlib.pyplot as plt

class Ridge:
    def __init__(self, input_dim, output_dim, alpha=1.0, fit_intercept=True, batch_size=32, device='cpu'):
        self.alpha = alpha
        self.fit_intercept = fit_intercept
        self.batch_size = batch_size
        self.device = device
        self.loss_history = []
        
        # Define the nn.Linear layer with the fit_intercept parameter controlling bias
        self.linear = nn.Linear(input_dim, output_dim, bias=fit_intercept).to(device)

    def fit(self, X, y, epochs=1000, lr=0.01):
        n_samples = X.shape[0]
        
        optimizer = optim.SGD(self.linear.parameters(), lr=lr)

        for epoch in range(epochs):
            indices = torch.randperm(n_samples)
            epoch_loss = 0.0
            num_batches = (n_samples + self.batch_size - 1) // self.batch_size

            for i in range(0, n_samples, self.batch_size):
                batch_indices = indices[i:i + self.batch_size]
                X_batch = X[batch_indices].to(self.device)
                y_batch = y[batch_indices].to(self.device)

                # Forward pass using the nn.Linear layer
                predictions = self.linear(X_batch)
                loss = self.loss(predictions, y_batch)
                epoch_loss += loss.item()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            avg_loss = epoch_loss / num_batches
            self.loss_history.append(avg_loss)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    def loss(self, predictions, y):
        # Mean squared error loss with L2 regularization (ridge penalty)
        mse_loss = torch.mean((predictions - y) ** 2)
        ridge_penalty = self.alpha * torch.sum(self.linear.weight ** 2)
        return mse_loss + ridge_penalty

    def predict(self, X):
        with torch.no_grad():
            return self.linear(X.to(self.device)).cpu().numpy()

    def plot_learning_curve(self):
        plt.plot(self.loss_history, label="Training Loss")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.title("Learning Curve")
        plt.legend()
        plt.show()

if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Generate synthetic data
    X_torch = torch.tensor(X_design.toarray()).float().to(device)  # Converts to dense PyTorch tensor
    y_torch = torch.tensor(y_data).float().to(device)

    # Instantiate and train the model
    torch_ridge = Ridge(input_dim=X_design.shape[1], output_dim=1, alpha=5, fit_intercept=True, batch_size=57777, device=device)
    torch_ridge.fit(X_torch, y_torch, epochs=300, lr=0.01)
    
    # Plot learning curve
    torch_ridge.plot_learning_curve()

    # Predictions
    torch_predictions = torch_ridge.predict(X_torch)
    print("Predictions shape:", torch_predictions.shape)


Epoch 1/300, Loss: 2.3984
Epoch 2/300, Loss: 1.6803
Epoch 3/300, Loss: 1.2727
Epoch 4/300, Loss: 1.0337
Epoch 5/300, Loss: 0.8863
Epoch 6/300, Loss: 0.7896
Epoch 7/300, Loss: 0.7220
Epoch 8/300, Loss: 0.6713
Epoch 9/300, Loss: 0.6320
Epoch 10/300, Loss: 0.5993
Epoch 11/300, Loss: 0.5720
Epoch 12/300, Loss: 0.5486
Epoch 13/300, Loss: 0.5283
Epoch 14/300, Loss: 0.5106
Epoch 15/300, Loss: 0.4945
Epoch 16/300, Loss: 0.4807
Epoch 17/300, Loss: 0.4683
Epoch 18/300, Loss: 0.4575
Epoch 19/300, Loss: 0.4476
Epoch 20/300, Loss: 0.4391
Epoch 21/300, Loss: 0.4314
Epoch 22/300, Loss: 0.4245
Epoch 23/300, Loss: 0.4189
Epoch 24/300, Loss: 0.4132
Epoch 25/300, Loss: 0.4085
Epoch 26/300, Loss: 0.4043
Epoch 27/300, Loss: 0.4005
Epoch 28/300, Loss: 0.3970
Epoch 29/300, Loss: 0.3946
Epoch 30/300, Loss: 0.3920
Epoch 31/300, Loss: 0.3892
Epoch 32/300, Loss: 0.3874
Epoch 33/300, Loss: 0.3854
Epoch 34/300, Loss: 0.3842
Epoch 35/300, Loss: 0.3824
Epoch 36/300, Loss: 0.3811
Epoch 37/300, Loss: 0.3798
Epoch 38/3

In [70]:
import torch
import torch.optim as optim
import torch.nn as nn
import matplotlib.pyplot as plt

class RidgeDeconvolution:
    def __init__(self, input_dim, output_dim, alpha=1.0, fit_intercept=True, batch_size=32, device='cpu'):
        self.alpha = alpha
        self.fit_intercept = fit_intercept
        self.batch_size = batch_size
        self.device = device
        self.loss_history = []
        
        # Define the linear layer with an optional intercept (bias term)
        self.linear = nn.Linear(input_dim, output_dim, bias=fit_intercept).to(device)
        
        # Variables to store mean and std for normalization
        self.X_mean = None
        self.X_std = None
        self.y_mean = None
        self.y_std = None

    def normalize_data(self, X, y):
        # Compute mean and std for normalization
        self.X_mean = X.mean(dim=0, keepdim=True)
        self.X_std = X.std(dim=0, keepdim=True)
        self.y_mean = y.mean(dim=0, keepdim=True)
        self.y_std = y.std(dim=0, keepdim=True)

        # Normalize X and y
        X_normalized = (X - self.X_mean) / (self.X_std + 1e-8)  # Avoid division by zero
        y_normalized = (y - self.y_mean) / (self.y_std + 1e-8)
        
        return X_normalized, y_normalized

    def fit(self, X, y, epochs=1000, lr=0.01):
        # Normalize the data
        X, y = self.normalize_data(X, y)
        n_samples = X.shape[0]
        
        optimizer = optim.SGD(self.linear.parameters(), lr=lr)

        for epoch in range(epochs):
            # Shuffle indices at the start of each epoch
            indices = torch.randperm(n_samples)
            epoch_loss = 0.0
            num_batches = (n_samples + self.batch_size - 1) // self.batch_size

            for i in range(0, n_samples, self.batch_size):
                batch_indices = indices[i:i + self.batch_size]
                X_batch = X[batch_indices].to(self.device)
                y_batch = y[batch_indices].to(self.device)

                # Forward pass using the linear layer
                predictions = self.linear(X_batch)
                loss = self.loss(predictions, y_batch)
                epoch_loss += loss.item()

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            avg_loss = epoch_loss / num_batches
            self.loss_history.append(avg_loss)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    def loss(self, predictions, y):
        mse_loss = torch.mean((predictions - y) ** 2)
        ridge_penalty = self.alpha * torch.sum(self.linear.weight ** 2)
        return mse_loss + ridge_penalty

    def predict(self, X):
        # Normalize the input X
        X_normalized = (X - self.X_mean) / (self.X_std + 1e-8)
        predictions_normalized = self.linear(X_normalized)
        
        # Rescale predictions to the original scale
        predictions = predictions_normalized * (self.y_std + 1e-8) + self.y_mean
        return predictions.detach().cpu().numpy()

    def plot_learning_curve(self):
        plt.plot(self.loss_history, label="Training Loss")
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.title("Learning Curve")
        plt.legend()
        plt.show()

if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Generate or load your design matrix for deconvolution (X_design)
    # and your response vector/matrix (y_data), ensuring that X_design 
    # properly reflects event separation with a Toeplitz structure if needed.
    X_torch = torch.tensor(X_design.toarray()).float().to(device)  # Convert to dense PyTorch tensor
    y_torch = torch.tensor(y_data).float().to(device)

    # Instantiate and train the deconvolution Ridge model
    torch_ridge = RidgeDeconvolution(input_dim=X_design.shape[1], output_dim=1, alpha=5, fit_intercept=True, batch_size=57777, device=device)
    torch_ridge.fit(X_torch, y_torch, epochs=200, lr=0.002)
    
    # Plot learning curve
    torch_ridge.plot_learning_curve()

    # Predictions
    torch_predictions = torch_ridge.predict(X_torch)
    print("Predictions shape:", torch_predictions.shape)


Epoch 1/200, Loss: 2.9220
Epoch 2/200, Loss: 2.7104
Epoch 3/200, Loss: 2.5479
Epoch 4/200, Loss: 2.3832
Epoch 5/200, Loss: 2.2453
Epoch 6/200, Loss: 2.1170
Epoch 7/200, Loss: 1.9737
Epoch 8/200, Loss: 1.8969
Epoch 9/200, Loss: 1.7949
Epoch 10/200, Loss: 1.6930
Epoch 11/200, Loss: 1.6139
Epoch 12/200, Loss: 1.5259
Epoch 13/200, Loss: 1.4811
Epoch 14/200, Loss: 1.4253
Epoch 15/200, Loss: 1.3698
Epoch 16/200, Loss: 1.3283
Epoch 17/200, Loss: 1.2902
Epoch 18/200, Loss: 1.2346
Epoch 19/200, Loss: 1.2030
Epoch 20/200, Loss: 1.1763
Epoch 21/200, Loss: 1.1166
Epoch 22/200, Loss: 1.1292
Epoch 23/200, Loss: 1.0754
Epoch 24/200, Loss: 1.0562
Epoch 25/200, Loss: 1.0324
Epoch 26/200, Loss: 1.0226
Epoch 27/200, Loss: 1.0083
Epoch 28/200, Loss: 0.9868
Epoch 29/200, Loss: 0.9730
Epoch 30/200, Loss: 0.9719
Epoch 31/200, Loss: 0.9555
Epoch 32/200, Loss: 0.9364
Epoch 33/200, Loss: 0.9387
Epoch 34/200, Loss: 0.9257
Epoch 35/200, Loss: 0.9262
Epoch 36/200, Loss: 0.9153
Epoch 37/200, Loss: 0.8846
Epoch 38/2

In [9]:
plt.plot(torch_ridge.linear.weight.detach().cpu().numpy().T)
plt.show()

In [52]:
# Revised PyTorch Ridge Model initialization
torch_ridge = Ridge(
    input_dim=X_design.shape[1], 
    output_dim=y_data.shape[1], 
    alpha=50,  # Smaller alpha closer to sklearn's regularization
    fit_intercept=False, 
    batch_size=500000,  # Smaller batch size for stable updates
    device=device
)

# Train with more epochs and a smaller learning rate
torch_ridge.fit(
    X_design, 
    y_data, 
    epochs=60,       # More epochs to ensure convergence
    lr=0.05,         # Lower learning rate for better precision
    momentum=0.9,     # Increased momentum for smoother updates
    scheduler_step_size=100, 
    scheduler_gamma=0.95  # Slower decay for the learning rate
)

# Plot learning curve to assess convergence
torch_ridge.plot_learning_curve()

# Extract and assign coefficients for comparison
coeffs = torch_ridge.linear.weight.detach().cpu().numpy()
rERP_model.coef_ = coeffs


TypeError: Ridge.fit() got an unexpected keyword argument 'momentum'

In [20]:
# Updated PyTorch Ridge Model initialization
torch_ridge = Ridge(
    input_dim=X_design.shape[1], 
    output_dim=y_data.shape[1], 
    alpha=5,  
    fit_intercept=False, 
    batch_size=500000,  # 10x larger batch size
    device=device
)

# Train with increased learning rate and reduced epochs
torch_ridge.fit(
    X_design, 
    y_data, 
    epochs=150,           # Fewer epochs due to larger batch size
    lr=0.005,              # Increased learning rate
    momentum=0.5,         # Reduced momentum for stability
    scheduler_step_size=0, 
    scheduler_gamma=0  # Slower decay in learning rate
)

# Plot learning curve to assess convergence
torch_ridge.plot_learning_curve()


TypeError: Ridge.fit() got an unexpected keyword argument 'momentum'

In [8]:
rERP_model.plot_coefs(top_topos=False)
plt.show()

Need more than one channel to make topography for eeg. Disabling interactivity.
Need more than one channel to make topography for eeg. Disabling interactivity.
Need more than one channel to make topography for eeg. Disabling interactivity.


In [16]:
plt.plot(coeffs.T)